## Case study: étude de l'offre et du positionnement de Freeglisse

**_Contexte_**

Olist aimerait désormais **augmenter son offre sur le segment du sport**, en incitant des marketplaces d'articles de sport à rejoindre la plateforme. Pour affiner sa stratégie vis-à-vis de ce type de vendeurs, Olist a besoin **d'analyser leur offre et leur positionnement**, pour mieux connaitre le marché sur lequel ils évoluent.

En particulier, les Sales d'Olist s'intéressent à **FreeGlisse**, une plateforme de vente en ligne de matériel de ski d'occasion. Leur ojectif est d'emmagasiner le plus d’informations possibles à partir de ce que FreeGlisse affiche sur son site web : https://freeglisse.com/fr/12-ski-occasion
<br><br>

**_Instructions_**

Olist voudrait se concentrer dans un premier temps sur le segment des **skis**. Les informations à étudier incluent : **le nombre d'annonces sur les skis**, les **prix** pratiqués, la **répartition de l’offre entre les différentes sous-catégories**, eventuellement la **profondeur des stocks** (= le nombre de produits disponibles, ou le nombre de tailles disponibles par produit), et tout ce que vous pensez qui serait **intéressant pour analyser le positionnement marché de FreeGlisse**.

Ces informations seraient d’autant plus intéressantes si elles étaient étudiées **par type de ski** (indiqué dans la Fiche Technique de chaque ski). N'hésitez pas à effectuer les regroupements qui vous semblent pertinents, à faire des analyses quantitatives et qualitatives, en utilisant **pandas** et en faisant de beaux **graphiques**. Pour les graphiques, vous utiliserez **Power BI**.

Côté technique, 2 compétences pourront être mises en oeuvre: le **crawling** entre les différentes pages du site, pour observer l’ensemble des offres de skis d’occasion (ici, pas besoin de selenium - des techniques plus simples seront possibles), et le **scraping des pages individuelles**, avec différentes infos à aller chercher.

Pour finir, l'objectif de ce cas est aussi de tirer de ces analyses des **insights business**, en particulier des **conclusions sur le positionnement commercial de FreeGlisse et les spécificités qu'il pourrait apporter à Olist**.
<br><br>

Pour extraire efficacement les informations des pages HTML, il est crucial de choisir les **bonnes méthodes de sélection** des éléments. Les méthodes **find()** et **find_all()** de BeautifulSoup sont couramment utilisées pour trouver des éléments dans un document HTML. Cependant, les méthodes **.select()** et **.select_one()** offrent une alternative plus **puissante et flexible**.

**_Présentation_**

En groupe, vous devrez **analyser les informations issues du scraping**, puis en fin de projet **présenter en environ 5 minutes** votre analyse à des Product Managers d'Olist (jurys de Databirdies), sous forme de **dashboard PowerBI**. L'essentiel est que la présentation soit **visuelle et intelligible par des stakeholders non-tech**.

### Utilité des Méthodes .select() et .select_one()

**.select():**

Cette méthode permet de sélectionner des éléments en utilisant des sélecteurs CSS, offrant ainsi une manière plus précise et flexible de cibler des éléments spécifiques dans le HTML

**.select_one():**

Cette méthode est similaire à .select(), mais elle ne retourne qu'un seul élément, le premier correspondant au sélecteur CSS fourni. C'est pratique lorsque vous savez qu'il n'y a qu'un seul élément correspondant au sélecteur ou lorsque vous souhaitez obtenir un élément spécifique sans avoir à parcourir une liste.

#### Avantages des Méthodes .select() et .select_one()

**_Flexibilité et Précision_**

Les sélecteurs CSS utilisés dans .select() permettent une sélection plus fine et plus flexible des éléments, surtout lorsque vous avez des structures HTML complexes.

**_Performances_**

Pour des opérations de sélection plus complexes ou spécifiques, .select() et .select_one() peuvent être plus performantes que find() et find_all(), car elles exploitent les sélecteurs CSS natifs qui sont optimisés pour ces tâches.

**_Syntaxe Moderne_**

L'utilisation des sélecteurs CSS est généralement plus intuitive et moderne, ce qui peut rendre le code plus lisible et facile à maintenir.

### C'est l'heure pour vous de scraper !

In [ ]:
import requests
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import json
from bs4 import BeautifulSoup
import re
import time
from datetime import datetime
from ast import literal_eval
from itertools import chain

In [ ]:
## BONJOUR DATA BIRD !!!!! coucou la team !!!

Information à récupérer :
*   Nombres d'annonces
*   Prix pratiqué
*   Repartition de l'offre entre les différents sous catégories
*   Profodeur du stock (nbr de produits dispo et taille par produits)
*   Marché du ski (haut de gamme ? / Particulier ?)
*   Type de ski

Constater après analyse que le ski d'occasion est plutot freestyle, que c'est plus pour femme ou hommes, etc...


In [ ]:
## DataFrame Principal (All Products)
## Fonction

def get_all_pages_general():

  urls = []
  page_number = 1
  for i in range (29):
    i = f"https://freeglisse.com/fr/12-ski-occasion?page={page_number}"
    page_number +=1
    urls.append(i)
  return urls

## Boucle + Dataframe
liste_nom = []
liste_ID = []
liste_url =[]
for x in get_all_pages_general():# boucle pour recuperer chaque url
  request = requests.get(x) # Recherche sur une page web
  soup = BeautifulSoup(request.content, "html.parser") # code html brute
  test_product = soup.find_all("div", class_ = "product-meta") # on selectionne le label "product meta" pour avoir chaque annonce
  for y in test_product: # boucle pour aller selectionner dans une annonce
    url = y.find("a").get("href")
    liste_url.append(url)
    ID = y.find("input", {"name": "id_product"}).get("value")
    liste_ID.append(ID) # ressort l'id
    nom = y.find("h2").text
    liste_nom.append(nom) # ressort le nom
df_general_ski = pd.DataFrame({"url" : liste_url, "ID": liste_ID, "Nom": liste_nom}) # creation du dataframe
df_general_ski

In [ ]:
df_general_ski.to_csv("df_general_ski.csv")

In [ ]:
df_general_ski.info()

In [ ]:
# boucle pour URL

def get_all_pages():

  urls = []
  page_number = 1
  for i in range (30):
    i = f"https://freeglisse.com/fr/12-ski-occasion?page={page_number}"
    page_number +=1
    urls.append(i)
  return urls


In [ ]:
## Scrapping Qualité A page 1

Gliss_A = requests.get("https://freeglisse.com/fr/12-ski-occasion/s-1/etat_du_materiel-qualite_a?page=1")
soup_A = BeautifulSoup(Gliss_A.content, "html.parser")
ski_A = soup_A.find_all('div',class_='product-meta')
print(ski_A)

In [ ]:
# Creation de la fonction qui retourne tous nos URL de qualité A => def variable qualite A

def qualité_A_all():

  urls = []
  page_number = 1
  for i in range (14):
    i = f"https://freeglisse.com/fr/12-ski-occasion/s-1/etat_du_materiel-qualite_a?page={page_number}"
    page_number +=1
    urls.append(i)
  return urls

In [ ]:
# Boucle DATAFRAME qualité A pour toutes les pages de sky d'aoccasion BRAVOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO
liste_qualite_A = []
liste_prix = []
liste_nom = []
liste_ID = []
liste_url =[]
for x in qualité_A_all():# boucle pour recuperer chaque url dans la liste qualité A => toute nos annonces
  request = requests.get(x) # Recherche sur une page web dans qualite A si code valide
  soup = BeautifulSoup(request.content, "html.parser") # code html brute
  test_product = soup.find_all("div", class_ = "product-meta") # on selection product meta pour avoir chaque annonce
  for y in test_product: # boucle pour aller selectionner dans une annonce
    url = y.find("a").get("href")
    liste_url.append(url)
    ID = y.find("input", {"name": "id_product"}).get("value")
    liste_ID.append(ID) # ressort l'id
    nom = y.find("h2").text
    liste_nom.append(nom) # ressort le nom
    prix = y.find("span", class_ ="price").text.strip()
    liste_prix.append(prix) # ressort le prix
    Qualite = y.find("span", class_ ="radio-label").text
    liste_qualite_A.append(Qualite) # ressort la qualite
result = pd.DataFrame({"url" : liste_url, "ID": liste_ID, "Nom": liste_nom, "Prix": liste_prix, "Qualité": liste_qualite_A})
print(result) # creation du dataframe

In [ ]:
# Transformation en CSV

result.to_csv("qualite_A.csv", encoding = "utf-8")

In [ ]:
# Creation de la fonction qui retourne tous nos URL de qualité B => def variable qualite B

def qualité_B_all():

  urls = []
  page_number = 1
  for i in range (15):
    i = f"https://freeglisse.com/fr/12-ski-occasion/s-1/etat_du_materiel-qualite_b?page={page_number}"
    page_number +=1
    urls.append(i)
  return urls

In [ ]:
# Boucle DATAFRAME qualité B pour toutes les pages de sky d'aoccasion BRAVOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO
liste_qualite_B = []
liste_prix = []
liste_nom = []
liste_ID = []
liste_url =[]
for x in qualité_B_all():# boucle pour recuperer chaque url dans la liste qualité B => toute nos annonces
  request = requests.get(x) # Recherche sur une page web dans qualite A si code valide
  soup = BeautifulSoup(request.content, "html.parser") # code html brute
  test_product = soup.find_all("div", class_ = "product-meta") # on selection product meta pour avoir chaque annonce
  for y in test_product: # boucle pour aller selectionner dans une annonce
    url = y.find("a").get("href")
    liste_url.append(url)
    ID = y.find("input", {"name": "id_product"}).get("value")
    liste_ID.append(ID) # ressort l'id
    nom = y.find("h2").text
    liste_nom.append(nom) # ressort le nom
    prix = y.find("span", class_ ="price").text.strip()
    liste_prix.append(prix) # ressort le prix
    Qualite = y.find("span", class_ ="radio-label").text
    liste_qualite_B.append(Qualite) # ressort la qualite
result_B = pd.DataFrame({"url" : liste_url, "ID": liste_ID, "Nom": liste_nom, "Prix": liste_prix, "Qualité": liste_qualite_B})
print(result_B) # creation du dataframe

In [ ]:
result_B ## Dataframe qualité B

In [ ]:
# Creation de la fonction qui retourne tous nos URL de qualité C => def variable qualite C

def qualité_C_all():

  urls = []
  page_number = 1
  for i in range (6):
    i = f"https://freeglisse.com/fr/12-ski-occasion/s-1/etat_du_materiel-qualite_c?page={page_number}"
    page_number +=1
    urls.append(i)
  return urls

In [ ]:
# Boucle DATAFRAME qualité C pour toutes les pages de sky d'aoccasion BRAVOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO
liste_qualite_C = []
liste_prix = []
liste_nom = []
liste_ID = []
liste_url =[]
for x in qualité_C_all():# boucle pour recuperer chaque url dans la liste qualité C => toute nos annonces
  request = requests.get(x) # Recherche sur une page web dans qualite C si code valide
  soup = BeautifulSoup(request.content, "html.parser") # code html brute
  test_product = soup.find_all("div", class_ = "product-meta") # on selection product meta pour avoir chaque annonce
  for y in test_product: # boucle pour aller selectionner dans une annonce
    url = y.find("a").get("href")
    liste_url.append(url)
    ID = y.find("input", {"name": "id_product"}).get("value")
    liste_ID.append(ID) # ressort l'id
    nom = y.find("h2").text
    liste_nom.append(nom) # ressort le nom
    prix = y.find("span", class_ ="price").text.strip()
    liste_prix.append(prix) # ressort le prix
    Qualite = y.find("span", class_ ="radio-label").text
    liste_qualite_C.append(Qualite) # ressort la qualite
result_C = pd.DataFrame({"url" : liste_url, "ID": liste_ID, "Nom": liste_nom, "Prix": liste_prix, "Qualité": liste_qualite_C})
print(result_C) # creation du dataframe

In [ ]:
# Transformation Qualité en full Qualité C
result_C["Qualité"] = result_C["Qualité"].replace("Qualité A", "C")
result_C["Qualité"] = result_C["Qualité"].replace("Qualité B", "C")
result_C["Qualité"] = result_C["Qualité"].replace("Qualité C", "C")
result_C

In [ ]:
# Création Dataframe Ski Qualité
df = pd.concat([result, result_B, result_C])
df.reset_index(drop=True, inplace=True)
df["Qualité"] = df["Qualité"].replace("Qualité A", "A")
df["Qualité"] = df["Qualité"].replace("Qualité B", "B")
df["Qualité"] = df["Qualité"].replace("Qualité C", "C")
df_ski_qualite = df

In [ ]:
df[df["ID"] == "20398"]

In [ ]:
df_ski_qualite.to_csv("df_ski_qualite.csv", encoding = "utf-8")

In [ ]:
df_ski_qualite.info()

** TYPE DE SKI (SCRAPPING) : 6 **

In [ ]:
# type de ski

# Creation de la fonction qui retourne tous nos URL de occasion femme

def occasion_femme_all():

  urls = []
  page_number = 1

  for i in range (6):
    i = f"https://freeglisse.com/fr/12-ski-occasion/s-1/types_de_ski-ski_occasion_femme?page={page_number}"
    page_number +=1
    urls.append(i)
  return urls



In [ ]:
# Boucle DATAFRAME occasion femme pour toutes les pages de sky d'aoccasion
liste_nom = []
liste_ID = []
liste_url = []
for x in occasion_femme_all():# boucle pour recuperer chaque url dans la liste occasion femme => toute nos annonces
  request = requests.get(x) # Recherche sur une page web dans occasion femme si code valide
  soup = BeautifulSoup(request.content, "html.parser") # code html brute
  test_product = soup.find_all("div", class_ = "product-meta") # on selection product meta pour avoir chaque annonce
  for y in test_product: # boucle pour aller selectionner dans une annonce
    url = y.find("a").get("href")
    liste_url.append(url)
    ID = y.find("input", {"name": "id_product"}).get("value")
    liste_ID.append(ID) # ressort l'id
    nom = y.find("h2").text
    liste_nom.append(nom) # ressort le nom

result_occasion_femme = pd.DataFrame({"url" : liste_url, "ID": liste_ID, "Nom": liste_nom})
print(result_occasion_femme) # creation du dataframe

In [ ]:
# doublon id  => avec duplicate sum = 22 doublons
result_occasion_femme['ID'].value_counts()

In [ ]:
# ajouter une colonnes categorie occ femme
result_occasion_femme['Categorie'] = 'occasion_femme'
print(result_occasion_femme)

In [ ]:
# type de ski

# Creation de la fonction qui retourne tous nos URL de occasion adulte

def occasion_adulte_all():

  urls = []
  page_number = 1

  for i in range (11):
    i = f"https://freeglisse.com/fr/12-ski-occasion/s-1/types_de_ski-ski_occasion_adulte?page={page_number}"
    page_number +=1
    urls.append(i)
  return urls

In [ ]:
# Boucle DATAFRAME de occasion adulte pour toutes les pages de sky d'occasion
liste_nom = []
liste_ID = []
liste_url = []
for x in occasion_adulte_all():# boucle pour recuperer chaque url dans la liste occasion adulte => toute nos annonces
  request = requests.get(x) # Recherche sur une page web dans qualite C si code valide
  soup = BeautifulSoup(request.content, "html.parser") # code html brute
  test_product = soup.find_all("div", class_ = "product-meta") # on selection product meta pour avoir chaque annonce
  for y in test_product: # boucle pour aller selectionner dans une annonce
    url = y.find("a").get("href")
    liste_url.append(url)
    ID = y.find("input", {"name": "id_product"}).get("value")
    liste_ID.append(ID) # ressort l'id
    nom = y.find("h2").text
    liste_nom.append(nom) # ressort le nom

result_occasion_adulte = pd.DataFrame({"url" : liste_url, "ID": liste_ID, "Nom": liste_nom})
print(result_occasion_adulte) # creation du dataframe

In [ ]:
# doublon id  => avec duplicate sum = 81 doublons
result_occasion_adulte['ID'].value_counts()

In [ ]:
result_occasion_adulte['ID'].duplicated().sum()

In [ ]:
# ajouter une colonnes categorie occ adulte
result_occasion_adulte['Categorie'] = 'occasion_adulte'
print(result_occasion_adulte)

In [ ]:
# type de ski

# Creation de la fonction qui retourne tous nos URL de occasion ski_occasion_adulte_freeride_et_freestyle

def ski_occasion_adulte_freeride_et_freestyle_all():

  urls = []
  page_number = 1

  for i in range (3):
    i = f"https://freeglisse.com/fr/12-ski-occasion/s-1/types_de_ski-ski_occasion_adulte_freeride_et_freestyle?page={page_number}"
    page_number +=1
    urls.append(i)
  return urls

In [ ]:
# Boucle DATAFRAME de occasion ski_occasion_adulte_freeride_et_freestyle pour toutes les pages de sky d'occasion
liste_nom = []
liste_ID = []
liste_url = []
for x in ski_occasion_adulte_freeride_et_freestyle_all():# boucle pour recuperer chaque url dans la liste occasion ski_occasion_adulte_freeride_et_freestyle_all
  request = requests.get(x) # Recherche sur une page web dans qualite ki_occasion_adulte_freeride_et_freestyle_all si code valide
  soup = BeautifulSoup(request.content, "html.parser") # code html brute
  test_product = soup.find_all("div", class_ = "product-meta") # on selection product meta pour avoir chaque annonce
  for y in test_product: # boucle pour aller selectionner dans une annonce
    url = y.find("a").get("href")
    liste_url.append(url)
    ID = y.find("input", {"name": "id_product"}).get("value")
    liste_ID.append(ID) # ressort l'id
    nom = y.find("h2").text
    liste_nom.append(nom) # ressort le nom

result_ski_occasion_adulte_freeride_et_freestyle = pd.DataFrame({"url" : liste_url, "ID": liste_ID, "Nom": liste_nom})
print(result_ski_occasion_adulte_freeride_et_freestyle) # creation du dataframe

In [ ]:
# doublon id  => avec duplicate sum = 4 doublons
result_ski_occasion_adulte_freeride_et_freestyle['ID'].duplicated().sum()

In [ ]:
# ajouter une colonnes categorie result_ski_occasion_adulte_freeride_et_freestyl
result_ski_occasion_adulte_freeride_et_freestyle['Categorie'] = 'adulte freeride/freestyle'
print(result_ski_occasion_adulte_freeride_et_freestyle)

In [ ]:
# type de ski

# Creation de la fonction qui retourne tous nos URL de occasion ski_occasion_junior

def ski_occasion_junior_all():

  urls = []
  page_number = 1

  for i in range (10):
    i = f"https://freeglisse.com/fr/12-ski-occasion/s-1/types_de_ski-ski_occasion_junior?page={page_number}"
    page_number +=1
    urls.append(i)
  return urls

In [ ]:
# Boucle DATAFRAME de occasion ski_occasion_junior pour toutes les pages de sky d'occasion
liste_nom = []
liste_ID = []
liste_url = []
for x in ski_occasion_junior_all():# boucle pour recuperer chaque url dans la liste occasion ski_occasion_junior
  request = requests.get(x) # Recherche sur une page web dans qualite ki_occasion_occasion ski_occasion_junior si code valide
  soup = BeautifulSoup(request.content, "html.parser") # code html brute
  test_product = soup.find_all("div", class_ = "product-meta") # on selection product meta pour avoir chaque annonce
  for y in test_product: # boucle pour aller selectionner dans une annonce
    url = y.find("a").get("href")
    liste_url.append(url)
    ID = y.find("input", {"name": "id_product"}).get("value")
    liste_ID.append(ID) # ressort l'id
    nom = y.find("h2").text
    liste_nom.append(nom) # ressort le nom

result_ski_occasion_junior = pd.DataFrame({"url" : liste_url, "ID": liste_ID, "Nom": liste_nom})
print(result_ski_occasion_junior) # creation du dataframe

In [ ]:
# doublon id  => avec duplicate sum = 32 doublons
result_ski_occasion_junior['ID'].duplicated().sum()

In [ ]:
# ajouter une colonnes categorie result_ski_occasion_adulte_freeride_et_freestyle
result_ski_occasion_junior['Categorie'] = 'junior'
print(result_ski_occasion_junior)

In [ ]:
# type de ski

# Creation de la fonction qui retourne tous nos URL de occasion ski_occasion_rando_telemark

def ski_occasion_rando_telemark_all():

  urls = []
  page_number = 1

  for i in range (10):
    i = f"https://freeglisse.com/fr/12-ski-occasion/s-1/types_de_ski-ski_occasion_randonnee_telemark?page={page_number}"
    page_number +=1
    urls.append(i)
  return urls

In [ ]:
# Boucle DATAFRAME de occasion ski_occasion_rando_telemark pour toutes les pages de sky d'occasion
liste_nom = []
liste_ID = []
liste_url = []
for x in ski_occasion_rando_telemark_all():
  request = requests.get(x)
  soup = BeautifulSoup(request.content, "html.parser")
  test_product = soup.find_all("div", class_ = "product-meta")
  for y in test_product:
    url = y.find("a").get("href")
    liste_url.append(url)
    ID = y.find("input", {"name": "id_product"}).get("value")
    liste_ID.append(ID)
    nom = y.find("h2").text
    liste_nom.append(nom)

result_ski_occasion_rando_telemark_all = pd.DataFrame({"url" : liste_url, "ID": liste_ID, "Nom": liste_nom})
print(result_ski_occasion_rando_telemark_all)

In [ ]:
# fusion de df type de ski

df_type_de_ski = pd.concat([result_occasion_femme, result_occasion_adulte, result_ski_occasion_adulte_freeride_et_freestyle, result_ski_occasion_junior, result_ski_occasion_rando_telemark_all, result_ski_occasion_ski_de_fond_all])
df_type_de_ski.reset_index(drop=True, inplace=True)
df_type_de_ski


In [ ]:
df_type_de_ski.to_csv("df_type_de_ski.csv", encoding = "utf-8")

# Type de pratique : **9**

In [ ]:
# doublon id  => avec duplicate sum = 0 doublons
result_ski_occasion_rando_telemark_all['ID'].duplicated().sum()

In [ ]:
# ajouter une colonnes categorie ski_occasion_rando_telemark
result_ski_occasion_rando_telemark_all['Categorie'] = 'rando/telemark'
print(result_ski_occasion_rando_telemark_all)

In [ ]:
# type de ski

# Creation de la fonction qui retourne tous nos URL de occasion ski de fond

def ski_occasion_ski_de_fond_all():

  urls = []
  page_number = 1

  for i in range (1):
    i = f"https://freeglisse.com/fr/12-ski-occasion/s-1/types_de_ski-ski_de_fond_occasion?page={page_number}"
    page_number +=1
    urls.append(i)
  return urls

In [ ]:
# Boucle DATAFRAME de occasion ski_occasion_ski_de_fond_all pour toutes les pages de sky d'occasion
liste_url = []
liste_nom = []
liste_ID = []
for x in ski_occasion_ski_de_fond_all():
  request = requests.get(x)
  soup = BeautifulSoup(request.content, "html.parser")
  test_product = soup.find_all("div", class_ = "product-meta")
  for y in test_product:
    url = y.find("a").get("href")
    liste_url.append(url)
    ID = y.find("input", {"name": "id_product"}).get("value")
    liste_ID.append(ID)
    nom = y.find("h2").text
    liste_nom.append(nom)

result_ski_occasion_ski_de_fond_all = pd.DataFrame({"url" : liste_url, "ID": liste_ID, "Nom": liste_nom})
print(result_ski_occasion_ski_de_fond_all)

In [ ]:
# doublon id  => avec duplicate sum = 0 doublons
result_ski_occasion_ski_de_fond_all['ID'].duplicated().sum()

In [ ]:
# ajouter une colonnes categorie result_ski_occasion_ski_de_fond_all
result_ski_occasion_ski_de_fond_all['Pratique'] = 'ski de fond'
print(result_ski_occasion_ski_de_fond_all)

In [ ]:
# type de pratique

# Creation de la fonction qui retourne tous nos URL de PRATIQUE  all moutain

def all_mountain():

  urls = []
  page_number = 1

  for i in range (6):
    i = f"https://freeglisse.com/fr/12-ski-occasion/s-1/votre_pratique-all_mountain?page={page_number}"
    page_number +=1
    urls.append(i)
  return urls

In [ ]:
# Boucle DATAFRAME PRATIQUE  all moutain
liste_url = []
liste_nom = []
liste_ID = []
for x in all_mountain():
  request = requests.get(x)
  soup = BeautifulSoup(request.content, "html.parser")
  test_product = soup.find_all("div", class_ = "product-meta")
  for y in test_product:
    url = y.find("a").get("href")
    liste_url.append(url)
    ID = y.find("input", {"name": "id_product"}).get("value")
    liste_ID.append(ID)
    nom = y.find("h2").text
    liste_nom.append(nom)

result_all_mountain = pd.DataFrame({"url" : liste_url, "ID": liste_ID, "Nom": liste_nom})
print(result_all_mountain)

In [ ]:
# ajouter une colonnes categorie PRATIQUE  all moutain

result_all_mountain['Pratique'] = 'mountain'
print(result_all_mountain)

In [ ]:
# type de ski

# Creation de la fonction qui retourne tous nos URL de occasion ski de fond

def polyvalent():

  urls = []
  page_number = 1

  for i in range (2):
    i = f"https://freeglisse.com/fr/12-ski-occasion/s-1/votre_pratique-polyvalent?page={page_number}"
    page_number +=1
    urls.append(i)
  return urls

In [ ]:
# Boucle DATAFRAME de occasion ski_occasion_ski_de_fond_all pour toutes les pages de sky d'occasion
liste_url = []
liste_nom = []
liste_ID = []
for x in polyvalent():
  request = requests.get(x)
  soup = BeautifulSoup(request.content, "html.parser")
  test_product = soup.find_all("div", class_ = "product-meta")
  for y in test_product:
    url = y.find("a").get("href")
    liste_url.append(url)
    ID = y.find("input", {"name": "id_product"}).get("value")
    liste_ID.append(ID)
    nom = y.find("h2").text
    liste_nom.append(nom)

result_polyvalent_all = pd.DataFrame({"url" : liste_url, "ID": liste_ID, "Nom": liste_nom})
print(result_polyvalent_all)

In [ ]:
# ajouter une colonnes categorie result_ski_occasion_ski_de_fond_all
result_polyvalent_all['Pratique'] = 'Polyvalent'
print(result_polyvalent_all)

In [ ]:
# type de ski

# Creation de la fonction qui retourne tous nos URL de occasion racing

def racing():

  urls = []
  page_number = 1

  for i in range (2):
    i = f"https://freeglisse.com/fr/12-ski-occasion/s-1/votre_pratique-racing?page={page_number}"
    page_number +=1
    urls.append(i)
  return urls

In [ ]:
# Boucle DATAFRAME de occasion ski_occasion_ski_de_fond_all pour toutes les pages de sky d'occasion
liste_url = []
liste_nom = []
liste_ID = []
for x in racing():
  request = requests.get(x)
  soup = BeautifulSoup(request.content, "html.parser")
  test_product = soup.find_all("div", class_ = "product-meta")
  for y in test_product:
    url = y.find("a").get("href")
    liste_url.append(url)
    ID = y.find("input", {"name": "id_product"}).get("value")
    liste_ID.append(ID)
    nom = y.find("h2").text
    liste_nom.append(nom)

result_racing_all = pd.DataFrame({"url" : liste_url, "ID": liste_ID, "Nom": liste_nom})
print(result_racing_all)

In [ ]:
# ajouter une colonnes categorie result_ski_occasion_ski_de_fond_all
result_racing_all['Pratique'] = 'Racing'
print(result_racing_all)

In [ ]:
# type de pratique

# Creation de la fonction qui retourne tous nos URL de occasion piste

def piste():

  urls = []
  page_number = 1

  for i in range (16):
    i = f"https://freeglisse.com/fr/12-ski-occasion/s-1/votre_pratique-piste?page={page_number}"
    page_number +=1
    urls.append(i)
  return urls

In [ ]:
# Boucle DATAFRAME de occasion result_piste_all pour toutes les pages de sky d'occasion
liste_url = []
liste_nom = []
liste_ID = []
for x in piste():
  request = requests.get(x)
  soup = BeautifulSoup(request.content, "html.parser")
  test_product = soup.find_all("div", class_ = "product-meta")
  for y in test_product:
    url = y.find("a").get("href")
    liste_url.append(url)
    ID = y.find("input", {"name": "id_product"}).get("value")
    liste_ID.append(ID)
    nom = y.find("h2").text
    liste_nom.append(nom)

result_piste_all = pd.DataFrame({"url" : liste_url, "ID": liste_ID, "Nom": liste_nom})
print(result_piste_all)

In [ ]:
# ajouter une colonnes categorie rresult_piste_all
result_piste_all['Pratique'] = 'Piste'
print(result_piste_all)

In [ ]:
# type de pratique

# Creation de la fonction qui retourne tous nos URL de freeride

def freeride():

  urls = []
  page_number = 1

  for i in range (2):
    i = f"https://freeglisse.com/fr/12-ski-occasion/s-1/votre_pratique-freeride?page={page_number}"
    page_number +=1
    urls.append(i)
  return urls

In [ ]:
# Boucle DATAFRAME de freeride pour toutes les pages de sky freeride
liste_url = []
liste_nom = []
liste_ID = []
for x in freeride():
  request = requests.get(x)
  soup = BeautifulSoup(request.content, "html.parser")
  test_product = soup.find_all("div", class_ = "product-meta")
  for y in test_product:
    url = y.find("a").get("href")
    liste_url.append(url)
    ID = y.find("input", {"name": "id_product"}).get("value")
    liste_ID.append(ID)
    nom = y.find("h2").text
    liste_nom.append(nom)

result_freeride_all = pd.DataFrame({"url" : liste_url, "ID": liste_ID, "Nom": liste_nom})
print(result_freeride_all)

In [ ]:
# ajouter une colonnes categorie rresult_piste_all
result_freeride_all['Pratique'] = 'freeride'
print(result_freeride_all)

In [ ]:
# type de pratique
# Creation de la fonction qui retourne tous nos URL de occasion piste

def freestyle():

  urls = []
  page_number = 1

  for i in range (2):
    i = f"https://freeglisse.com/fr/12-ski-occasion/s-1/votre_pratique-freestyle?page={page_number}"
    page_number +=1
    urls.append(i)
  return urls

In [ ]:
# Boucle DATAFRAME de freeride pour toutes les pages de freestyle
liste_url = []
liste_nom = []
liste_ID = []
for x in freestyle():
  request = requests.get(x)
  soup = BeautifulSoup(request.content, "html.parser")
  test_product = soup.find_all("div", class_ = "product-meta")
  for y in test_product:
    url = y.find("a").get("href")
    liste_url.append(url)
    ID = y.find("input", {"name": "id_product"}).get("value")
    liste_ID.append(ID)
    nom = y.find("h2").text
    liste_nom.append(nom)

result_freestyle_all = pd.DataFrame({"url" : liste_url, "ID": liste_ID, "Nom": liste_nom})
print(result_freestyle_all)

In [ ]:
# ajouter une colonnes categorie rresult_piste_all
result_freestyle_all['Pratique'] = 'freestyle'
print(result_freestyle_all)

In [ ]:
# type de pratique

# Creation de la fonction qui retourne tous nos URL de occasion piste

def alternatif():

  urls = []
  page_number = 1

  for i in range (1):
    i = f"https://freeglisse.com/fr/12-ski-occasion/s-1/votre_pratique-alternatif?page={page_number}"
    page_number +=1
    urls.append(i)
  return urls

In [ ]:
# Boucle DATAFRAME de freeride pour toutes les pages de alternatif
liste_url = []
liste_nom = []
liste_ID = []
for x in alternatif():
  request = requests.get(x)
  soup = BeautifulSoup(request.content, "html.parser")
  test_product = soup.find_all("div", class_ = "product-meta")
  for y in test_product:
    url = y.find("a").get("href")
    liste_url.append(url)
    ID = y.find("input", {"name": "id_product"}).get("value")
    liste_ID.append(ID)
    nom = y.find("h2").text
    liste_nom.append(nom)

result_alternatif_all = pd.DataFrame({"url" : liste_url, "ID": liste_ID, "Nom": liste_nom})
print(result_alternatif_all)

In [ ]:
# ajouter une colonnes categorie rresult_piste_all
result_freestyle_all['Pratique'] = 'alternatif'
print(result_freestyle_all)

In [ ]:
# type de pratique

# Creation de la fonction qui retourne tous nos URL de skating

def skating():

  urls = []
  page_number = 1

  for i in range (1):
    i = f"https://freeglisse.com/fr/12-ski-occasion/s-1/votre_pratique-skating?page={page_number}"
    page_number +=1
    urls.append(i)
  return urls

In [ ]:
# Boucle DATAFRAME de freeride pour toutes les pages de skating
liste_url = []
liste_nom = []
liste_ID = []
for x in skating():
  request = requests.get(x)
  soup = BeautifulSoup(request.content, "html.parser")
  test_product = soup.find_all("div", class_ = "product-meta")
  for y in test_product:
    url = y.find("a").get("href")
    liste_url.append(url)
    ID = y.find("input", {"name": "id_product"}).get("value")
    liste_ID.append(ID)
    nom = y.find("h2").text
    liste_nom.append(nom)

result_skating_all = pd.DataFrame({"url" : liste_url, "ID": liste_ID, "Nom": liste_nom})
print(result_skating_all)

In [ ]:
# ajouter une colonnes categorie rresult_piste_all
result_skating_all['Pratique'] = 'skating'
print(result_skating_all)

In [ ]:
# type de pratique

# Creation de la fonction qui retourne tous nos URL de randonnee

def randonnee():

  urls = []
  page_number = 1

  for i in range (1):
    i = f"https://freeglisse.com/fr/12-ski-occasion/s-1/votre_pratique-ski_de_randonnee?page={page_number}"
    page_number +=1
    urls.append(i)
  return urls

In [ ]:
# Boucle DATAFRAME de freeride pour toutes les pages de randonee
liste_url = []
liste_nom = []
liste_ID = []
for x in randonnee():
  request = requests.get(x)
  soup = BeautifulSoup(request.content, "html.parser")
  test_product = soup.find_all("div", class_ = "product-meta")
  for y in test_product:
    url = y.find("a").get("href")
    liste_url.append(url)
    ID = y.find("input", {"name": "id_product"}).get("value")
    liste_ID.append(ID)
    nom = y.find("h2").text
    liste_nom.append(nom)

result_randonnee_all = pd.DataFrame({"url" : liste_url, "ID": liste_ID, "Nom": liste_nom})
print(result_randonnee_all)

In [ ]:
# ajouter une colonnes categorie rresult_piste_all
result_randonnee_all['Pratique'] = 'randonnee'
print(result_randonnee_all)

In [ ]:
# fusion des df pratique

df_pratique = pd.concat([result_ski_occasion_ski_de_fond_all, result_all_mountain, result_polyvalent_all, result_racing_all, result_piste_all, result_freeride_all, result_freestyle_all, result_alternatif_all, result_skating_all, result_randonnee_all])
df_pratique.reset_index(drop=True, inplace=True)
df_pratique

In [ ]:
df_pratique["Pratique"].isna().sum()

In [ ]:
df_pratique = df_pratique.dropna()
df_pratique.reset_index(drop=True, inplace=True)
df_pratique.info()

In [ ]:
df_pratique.to_csv("df_pratique.csv", encoding = "utf-8")